# 🛡️ DeepShield V4-Universal — Cross-Manipulation Training

V3 learned **generated faces** (StyleGAN1, StyleGAN2/thispersondoesnotexist, diffusion) and scores them 0.97+. It has never seen a **face-swap**, so DFDC-style video deepfakes slip past it.

V4 adds DFDC face crops — both classes:
- **FAKE** frames teach the face-swap artefact family (blending seams, edge mismatch)
- **REAL** frames add authentic faces from ordinary cameras, lighting and compression. Our real class was FFHQ-only, which is why a pristine press portrait once scored 0.94 fake.

**Honest expectation:** ~75-85% on DFDC-style face-swaps. The $1M DFDC winner reached 82% on its own test set and ~65% on unseen deepfakes — cross-manipulation generalisation is an open research problem, not a bug in this notebook.

## Setup — two switches

Right panel → **Session options**: **Accelerator → GPU T4 x2**, **Internet → On**. Then **Run All**.

Everything downloads itself; attaching datasets via **+ Add Input** only saves time.

Outputs (right panel → **Output**): the trained model, training curves, confusion matrix, and `demo_samples.zip` — 24 blind images with a separate answer key for the live demo.

In [ ]:
# ── 1. Environment ───────────────────────────────────────────────
import torch, os, glob
print('PyTorch :', torch.__version__)
assert torch.cuda.is_available(), 'No GPU! Session options → Accelerator → GPU'
device = torch.device('cuda')
print('GPU     :', torch.cuda.get_device_name(0))

# /kaggle/working persists for the whole session (and into saved versions)
CKPT_DIR = '/kaggle/working'
RESUME_PATH = f'{CKPT_DIR}/resume.pth'
print('Checkpoints →', CKPT_DIR)

In [ ]:
# ── 2. Get datasets: attached inputs first, else download ────────
INPUT = '/kaggle/input'
attached = sorted(os.listdir(INPUT)) if os.path.isdir(INPUT) else []
print('Attached inputs:', attached or 'none — will download instead')

def find_root(base, marker_dirs):
    for root, dirs, _ in os.walk(base):
        if marker_dirs <= set(dirs):
            return root
    return None

def all_images(path):
    files = []
    for e in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        files += glob.glob(os.path.join(path, '**', e), recursive=True)
    return sorted(files)

def input_dir(*keywords):
    for name in attached:
        low = name.lower()
        if all(k in low for k in keywords):
            return os.path.join(INPUT, name)
    return None

def get_dataset(slug, *keywords):
    d = input_dir(*keywords)
    if d:
        return d, 'attached'
    import kagglehub
    return kagglehub.dataset_download(slug), 'downloaded'

# -- Base: 140k real-and-fake (SG1 fakes + FFHQ reals) — REQUIRED
base, how = get_dataset('xhlulu/140k-real-and-fake-faces', '140k')
DATA = find_root(base, {'train', 'valid', 'test'})
assert DATA, 'train/valid/test folders not found in the 140k dataset'
print(f'140k base   : {DATA}  ({how})')

# -- Generated-face families (tpdn required; others optional)
EXTRAS = {}
for key, slug, keywords in [
    ('tpdn',      'almightyj/person-face-dataset-thispersondoesnotexist', ('thispersondoesnotexist',)),
    ('sg2_hires', 'hyperclaw79/fakefaces',                                ('fakefaces',)),
    ('diffusion', 'mohannadaymansalah/stable-diffusion-dataaaaaaaaa',     ('diffusion',)),
]:
    try:
        d, how = get_dataset(slug, *keywords)
        EXTRAS[key] = all_images(d)
        print(f'{key:10s}: {len(EXTRAS[key]):6d} images  ({how})')
    except Exception as e:
        print(f'{key:10s}: SKIPPED ({type(e).__name__})')
assert EXTRAS.get('tpdn'), 'TPDN set unavailable — check Internet: On'

# -- NEW in V4: DFDC face crops (face-swap family), already 224x224
#    Labels live in metadata.csv: videoname,label(REAL/FAKE),original,split
import pandas as pd
DFDC_FAKE, DFDC_REAL = [], []
try:
    dfdc, how = get_dataset('dagnelies/deepfake-faces', 'deepfake', 'faces')
    meta_path = next(iter(glob.glob(os.path.join(dfdc, '**', 'metadata.csv'), recursive=True)), None)
    faces_dir = next((r for r, d, f in os.walk(dfdc) if len(glob.glob(os.path.join(r, '*.jpg'))) > 500), None)
    assert meta_path and faces_dir, 'metadata.csv or face folder not found'
    meta = pd.read_csv(meta_path)
    print('DFDC metadata columns:', list(meta.columns))
    name_col = 'videoname' if 'videoname' in meta.columns else meta.columns[0]
    have = set(os.listdir(faces_dir))
    for _, row in meta.iterrows():
        fn = str(row[name_col]).replace('.mp4', '.jpg')
        if fn not in have:
            continue
        p = os.path.join(faces_dir, fn)
        (DFDC_FAKE if str(row['label']).upper() == 'FAKE' else DFDC_REAL).append(p)
    print(f'dfdc      : {len(DFDC_FAKE)} fake + {len(DFDC_REAL)} real face crops  ({how})')
except Exception as e:
    print(f'dfdc      : SKIPPED ({type(e).__name__}: {e}) — V4 falls back to V3 data')

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────
BACKBONE       = 'large'    # V4 keeps the Large backbone
PER_CLASS      = 50000      # training images per class
VAL_PER_CLASS  = 2500
TPDN_HOLDOUT_N = 1000       # generated-face holdout (never trained on)
DFDC_HOLDOUT_N = 1000       # face-swap holdout   (never trained on)
DFDC_SHARE     = 0.35       # share of the fake budget given to face-swaps
EPOCHS         = 10
BATCH          = 128
LR             = 3e-4
IMG            = 224
SEED           = 42

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# ── 4. Build the cross-manipulation training set ─────────────────
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
import io

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class RandomJPEG:
    """Randomise the compression domain so the model cannot key on it."""
    def __init__(self, p=0.7, quality=(30, 95)):
        self.p, self.quality = p, quality
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            img.save(buf, 'JPEG', quality=random.randint(*self.quality))
            buf.seek(0)
            img = PILImage.open(buf).convert('RGB')
        return img

class RandomRescale:
    """Down-then-up scaling — removes resolution fingerprints."""
    def __init__(self, p=0.5, lo=0.5):
        self.p, self.lo = p, lo
    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size
            s = random.uniform(self.lo, 1.0)
            img = img.resize((max(32, int(w*s)), max(32, int(h*s))), PILImage.BILINEAR)
            img = img.resize((w, h), PILImage.BILINEAR)
        return img

class FixedJPEG:
    def __init__(self, quality=40):
        self.q = quality
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=self.q)
        buf.seek(0)
        return PILImage.open(buf).convert('RGB')

train_tf = transforms.Compose([
    RandomRescale(), RandomJPEG(),
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf   = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(),
                                transforms.Normalize(MEAN, STD)])
robust_tf = transforms.Compose([FixedJPEG(40), transforms.Resize((IMG, IMG)),
                                transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class FlatImages(Dataset):
    """(path, label) pairs; label 0 = fake, 1 = real."""
    def __init__(self, samples, tf):
        self.samples, self.tf = samples, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        return self.tf(PILImage.open(p).convert('RGB')), y

CLASSES = ['fake', 'real']   # index order must match the app
rng = random.Random(SEED)

def take(lst, n):
    lst = list(lst); rng.shuffle(lst); return lst[:n]

# --- holdouts carved out FIRST so they are never trained on ---
tpdn_all = list(EXTRAS['tpdn']); rng.shuffle(tpdn_all)
tpdn_holdout, tpdn_train = tpdn_all[:TPDN_HOLDOUT_N], tpdn_all[TPDN_HOLDOUT_N:]

dfdc_fake = list(DFDC_FAKE); rng.shuffle(dfdc_fake)
dfdc_real = list(DFDC_REAL); rng.shuffle(dfdc_real)
half = DFDC_HOLDOUT_N // 2
dfdc_holdout = [(p, 0) for p in dfdc_fake[:half]] + [(p, 1) for p in dfdc_real[:half]]
dfdc_fake, dfdc_real = dfdc_fake[half:], dfdc_real[half:]

# --- FAKE class: generated faces + a face-swap share ---
n_swap = int(PER_CLASS * DFDC_SHARE)
swap_part = take(dfdc_fake, n_swap)
gen_pool  = tpdn_train + list(EXTRAS.get('sg2_hires', [])) + list(EXTRAS.get('diffusion', []))
gen_part  = take(gen_pool, PER_CLASS - len(swap_part))
sg1 = all_images(os.path.join(DATA, 'train', 'fake'))
gen_part += take(sg1, PER_CLASS - len(swap_part) - len(gen_part))
fakes = swap_part + gen_part

# --- REAL class: FFHQ + real DFDC frames (diverse cameras/lighting) ---
real_swapsrc = take(dfdc_real, int(PER_CLASS * DFDC_SHARE))
ffhq = all_images(os.path.join(DATA, 'train', 'real'))
reals = real_swapsrc + take(ffhq, len(fakes) - len(real_swapsrc))

print(f'FAKE  : {len(swap_part)} face-swap + {len(gen_part)} generated = {len(fakes)}')
print(f'REAL  : {len(real_swapsrc)} DFDC real + {len(reals)-len(real_swapsrc)} FFHQ = {len(reals)}')
print(f'HOLDOUTS: TPDN {len(tpdn_holdout)} · DFDC {len(dfdc_holdout)}')

train_samples = [(p, 0) for p in fakes] + [(p, 1) for p in reals]
rng.shuffle(train_samples)

def per_class(folder, n, label, seed):
    files = all_images(folder); random.Random(seed).shuffle(files)
    return [(p, label) for p in files[:n]]

valid_samples = (per_class(f'{DATA}/valid/fake', VAL_PER_CLASS, 0, SEED)
               + per_class(f'{DATA}/valid/real', VAL_PER_CLASS, 1, SEED))
test_samples  = ([(p, 0) for p in all_images(f'{DATA}/test/fake')]
               + [(p, 1) for p in all_images(f'{DATA}/test/real')])
tpdn_samples  = [(p, 0) for p in tpdn_holdout]

mk = lambda s, tf, **kw: DataLoader(FlatImages(s, tf), batch_size=BATCH,
                                    num_workers=2, pin_memory=True, **kw)
train_dl  = DataLoader(FlatImages(train_samples, train_tf), batch_size=BATCH, shuffle=True,
                       num_workers=4, pin_memory=True, persistent_workers=True)
valid_dl  = mk(valid_samples, eval_tf)
robust_dl = mk(valid_samples, robust_tf)
tpdn_dl   = mk(tpdn_samples, eval_tf)
dfdc_dl   = mk(dfdc_holdout, eval_tf) if dfdc_holdout else None
test_dl   = mk(test_samples, eval_tf)


In [ ]:
# ── 5. Model: MobileNetV3 (large or small) ───────────────────────
from torchvision import models

def build_model(backbone):
    if backbone == 'large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    else:
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = torch.nn.Linear(m.classifier[3].in_features, 2)
    return m

model = build_model(BACKBONE).to(device)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'MobileNetV3-{BACKBONE} ready — {params:.1f}M parameters')

In [ ]:
# ── 6. Train — resume-safe, watching both holdouts ───────────────
import copy
from tqdm.auto import tqdm

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(train_dl), eta_min=1e-5)

@torch.no_grad()
def evaluate(dl):
    if dl is None:
        return float('nan')
    model.eval()
    correct = total = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

history = {'loss': [], 'val_acc': [], 'robust_acc': [], 'tpdn_acc': [], 'dfdc_acc': []}
best_score, best_state, start_epoch = 0.0, None, 1

# Resume: a killed session picks up from the last finished epoch
if os.path.exists(RESUME_PATH):
    ck = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    if ck.get('backbone') == BACKBONE:
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        history, best_score, best_state = ck['history'], ck['best_score'], ck['best_state']
        start_epoch = ck['epoch'] + 1
        print(f'RESUMED from epoch {ck["epoch"]} (best {best_score:.4f})')
    else:
        print('resume file is for a different backbone — starting fresh')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running = 0.0
    bar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS}')
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step(); scheduler.step()
        running += loss.item() * yb.size(0)
        bar.set_postfix(loss=f'{loss.item():.3f}')

    m = {'loss': running / len(train_samples), 'val_acc': evaluate(valid_dl),
         'robust_acc': evaluate(robust_dl), 'tpdn_acc': evaluate(tpdn_dl),
         'dfdc_acc': evaluate(dfdc_dl)}
    for k, v in m.items():
        history[k].append(v)
    print(f'Epoch {epoch}: loss {m["loss"]:.4f} · val {m["val_acc"]*100:.2f}% '
          f'· robust {m["robust_acc"]*100:.2f}% · TPDN {m["tpdn_acc"]*100:.2f}% '
          f'· DFDC {m["dfdc_acc"]*100:.2f}%')

    # Best = balanced across all three generalisation axes we can measure
    parts = [m['robust_acc'], m['tpdn_acc']] + ([m['dfdc_acc']] if m['dfdc_acc'] == m['dfdc_acc'] else [])
    score = sum(parts) / len(parts)
    if score > best_score:
        best_score, best_state = score, copy.deepcopy(model.state_dict())

    torch.save({'epoch': epoch, 'backbone': BACKBONE, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'history': history, 'best_score': best_score, 'best_state': best_state},
               RESUME_PATH)

model.load_state_dict(best_state)
print(f'\nBest combined score: {best_score:.4f}')

In [ ]:
# ── 7. Curves (report material) ──────────────────────────────────
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history['loss']) + 1)
a1.plot(ep, history['loss'], marker='o'); a1.set_title('Training loss'); a1.set_xlabel('epoch')
a2.plot(ep, [a*100 for a in history['val_acc']],    marker='o', color='green',  label='clean val')
a2.plot(ep, [a*100 for a in history['robust_acc']], marker='s', color='orange', label='robust val (q40)')
a2.plot(ep, [a*100 for a in history['tpdn_acc']],   marker='^', color='red',    label='TPDN holdout (generated)')
if history['dfdc_acc'] and history['dfdc_acc'][0] == history['dfdc_acc'][0]:
    a2.plot(ep, [a*100 for a in history['dfdc_acc']], marker='D', color='purple', label='DFDC holdout (face-swap)')
a2.set_title('Accuracy (%)'); a2.set_xlabel('epoch'); a2.legend(fontsize=8)
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
# ── 8. Final evaluation on every held-out set ────────────────────
from sklearn.metrics import confusion_matrix, classification_report

test_acc   = evaluate(test_dl)
tpdn_final = evaluate(tpdn_dl)
dfdc_final = evaluate(dfdc_dl)
print(f'140k TEST accuracy : {test_acc*100:.2f}%')
print(f'TPDN  holdout      : {tpdn_final*100:.2f}%   (generated faces)')
print(f'DFDC  holdout      : {dfdc_final*100:.2f}%   (face-swap — the V4 target)')

preds, trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc='confusion'):
        preds += model(xb.to(device)).argmax(1).cpu().tolist()
        trues += yb.tolist()
print(classification_report(trues, preds, target_names=CLASSES))
cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'V4-Universal ({test_acc*100:.1f}%)')
plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ── 9. Export for DeepShield ─────────────────────────────────────
CKPT = 'deepshield_mobilenetv3.pth'
torch.save({
    'arch':          f'mobilenet_v3_{BACKBONE}',
    'state_dict':    model.state_dict(),
    'classes':       CLASSES,
    'input_size':    IMG,
    'normalize':     {'mean': MEAN, 'std': STD},
    'val_accuracy':  round(history['val_acc'][-1] * 100, 2),
    'robust_val_accuracy': round(max(history['robust_acc']) * 100, 2),
    'tpdn_accuracy': round(tpdn_final * 100, 2),
    'dfdc_accuracy': round(dfdc_final * 100, 2),
    'test_accuracy': round(float(test_acc) * 100, 2),
    'trained_on':    ('V4-Universal: SG1 + TPDN/SG2 + diffusion + DFDC face-swap, '
                      f'{len(train_samples)} imgs, {EPOCHS} epochs, {BACKBONE}'),
}, CKPT)
print(f'Saved {CKPT} — {os.path.getsize(CKPT)/1e6:.1f} MB')

In [ ]:
# ── 10. Blind demo pack — 24 unseen images + answer key ──────────
# Anonymous filenames, ground truth in a separate file: hand the folder
# to an examiner, let them pick, then open the key.
import shutil

DEMO = 'demo_samples'
shutil.rmtree(DEMO, ignore_errors=True); os.makedirs(DEMO)

picks = []                                   # (path, truth, source)
gen_fake = [p for p in tpdn_holdout[:6]]
picks += [(p, 'FAKE', 'generated face') for p in gen_fake]
picks += [(p, 'FAKE', 'face-swap') for p, y in dfdc_holdout if y == 0][:6]
picks += [(p, 'REAL', 'real (DFDC)')  for p, y in dfdc_holdout if y == 1][:6]
picks += [(p, 'REAL', 'real (FFHQ)') for p in take(all_images(f'{DATA}/test/real'), 6)]

rng2 = random.Random(7); rng2.shuffle(picks)
key = []
for i, (p, truth, kind) in enumerate(picks, 1):
    out = f'sample_{i:02d}.jpg'
    PILImage.open(p).convert('RGB').save(f'{DEMO}/{out}', quality=92)
    key.append(f'{out}  ->  {truth:4s}  ({kind})')

with open(f'{DEMO}/ANSWER_KEY.txt', 'w') as f:
    f.write('DeepShield demo samples - ground truth\n(open only AFTER the demo)\n\n')
    f.write('\n'.join(key))

shutil.make_archive('demo_samples', 'zip', DEMO)
print(f'{len(picks)} blind samples across both manipulation types -> demo_samples.zip')

## ✅ Done

Right panel → **Output** → download:

1. **`deepshield_mobilenetv3.pth`** → save as `models\archive\v4_universal.pth`, then copy over `models\deepshield_mobilenetv3.pth` to go live. The backend hot-reloads and reads the architecture from the checkpoint.
2. **`demo_samples.zip`** → unzip into `training\demo_samples\`. 24 unseen images spanning generated faces, face-swaps and two kinds of real photo, with the answer key in a separate file.
3. **`training_curves.png`** + **`confusion_matrix.png`** → `training\results\`.

Then tell Claude **"v4 aa gaya"** for the head-to-head against V3.

> **Reading the numbers honestly:** TPDN measures generated faces, DFDC measures face-swaps, robust-val measures survival under compression. A drop on DFDC relative to TPDN is expected — face-swap detection is a harder, largely unsolved problem, and the \$1M DFDC winner scored 82% on its own test set and around 65% on unseen deepfakes.